In [1]:
#!/usr/bin/env python3
"""
Federal Reserve Speech & Testimony Scraper — One-Time Backfill
================================================================

Closes the gap left by the RSS feed's 15-entry limit. The RSS feed only
ever holds the ~15 most recent items, so anything that aged off the feed
between your last historical CSV entry (2026-02-16) and whenever the 15
feed items start was never scraped by TOOLS_automate_scrape.py.

Two source pages fill that gap, since the Fed doesn't offer one combined
archive:

  1. SPEECHES_ARCHIVE_HTML — the static yearly archive page
     (federalreserve.gov/newsevents/2026-speeches.htm), saved locally.
     No JS/Selenium needed; content is server-rendered. Lists every 2026
     speech.

  2. TESTIMONY_RESULTS_HTML — the filterable speeches-testimony.htm page
     (federalreserve.gov/newsevents/speeches-testimony.htm), saved locally
     after filtering to Type=Testimony and the date range 2/16/2026-9/7/2026.
     There is no yearly testimony archive equivalent to (1), but this filter
     page covers the same gap for testimony specifically.

Both pages are read from local HTML files (passed as paths below) rather
than fetched live, since this is a one-time run against files you already
saved/uploaded. Update the two paths in main() if you re-run this against
freshly saved copies.

After extracting (date, link) pairs from both pages, this script:
  - filters to links dated after the last date already in fed_speech.csv
  - skips anything already in .seen_links.txt
  - scrapes each remaining link with the existing scrape_speech_page()
    (imported from TOOLS_automate_scrape.py, so speech and testimony pages
    are parsed identically to how the incremental updater does it)
  - appends rows to fed_speech.csv and links to .seen_links.txt, in
    chronological order (oldest first) so ids stay in date order

Run this once, then let TOOLS_automate_scrape.py's hourly cron take over —
by the time this backfill finishes, nothing will have had a chance to age
off the RSS feed's 15-entry window.
"""

import datetime
import sys
import time
from pathlib import Path

from bs4 import BeautifulSoup

# Reuses everything from the incremental updater so both scripts parse
# individual speech/testimony pages identically and write the same CSV
# format.
from TOOLS_automate_scrape import (
    CSV_PATH,
    SEEN_LINKS_PATH,
    REQUEST_DELAY_SECONDS,
    scrape_speech_page,
    load_seen_links,
    append_seen_links,
    get_next_id,
    get_last_date_in_csv,
    normalize_date,
    append_rows_to_csv,
)


# ---------------------------------------------------------------------------
# Page-specific parsers
#
# These two pages use different markup from each other (and from the
# individual speech/testimony pages scrape_newsevents() already handles),
# but for backfill purposes only the (date, link) pairs matter — the actual
# title/speaker/content gets pulled from each individual page via
# scrape_speech_page(), same as the incremental updater does.
# ---------------------------------------------------------------------------

def parse_speeches_archive(html_path: Path) -> list[tuple[str, str]]:
    """
    Parses the static yearly archive page (e.g. 2026-speeches.htm).
    Markup: div.eventlist__time > <time>M/D/YYYY</time>, paired with a
    sibling div.eventlist__event whose first non-watchLive <a> is the
    speech link.
    """
    soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "html.parser")
    container = soup.find("div", class_=lambda c: c and "eventlist" in c.split())
    if container is None:
        print(f"  ! Could not find eventlist container in {html_path}", file=sys.stderr)
        return []

    out = []
    for time_div in container.find_all("div", class_=lambda c: c and "eventlist__time" in c.split()):
        row = time_div.find_parent("div", class_=lambda c: c and "row" in c.split())
        time_tag = time_div.find("time")
        event_div = row.find("div", class_=lambda c: c and "eventlist__event" in c.split())
        if not time_tag or not event_div:
            continue
        date = time_tag.get_text(strip=True)
        # The title link has no class; watchLive links do. Grabbing the
        # first non-watchLive <a> reliably gets the title link regardless
        # of whether a watchLive link is present before or after it.
        a = event_div.find("a", class_=lambda c: not c or "watchLive" not in c)
        if a and a.get("href"):
            out.append((date, a["href"]))
    return out


def parse_testimony_results(html_path: Path) -> list[tuple[str, str]]:
    """
    Parses the filtered speeches-testimony.htm results page (saved after
    filtering to Type=Testimony + a date range). Markup differs from the
    static archive: <time class="itemDate"> and <p class="itemTitle"><em>
    <a>...</a></em></p>.
    """
    soup = BeautifulSoup(html_path.read_text(encoding="utf-8"), "html.parser")
    container = soup.find("div", id="speech-results")
    if container is None:
        print(f"  ! Could not find #speech-results in {html_path}", file=sys.stderr)
        return []

    out = []
    for time_tag in container.find_all("time", class_="itemDate"):
        row = time_tag.find_parent("div", class_=lambda c: c and "row" in c.split())
        event_div = row.find("div", class_=lambda c: c and "eventlist__event" in c.split())
        if event_div is None:
            continue
        date = time_tag.get_text(strip=True)
        title_p = event_div.find("p", class_="itemTitle")
        a = title_p.find("a") if title_p else None
        if a and a.get("href"):
            out.append((date, a["href"]))
    return out


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    speeches_html = Path("samples/Federal Reserve Board - 2026 Speeches.html")
    testimony_html = Path("samples/Federal Reserve Board - Testimonies Only.html")

    speech_items = parse_speeches_archive(speeches_html)
    testimony_items = parse_testimony_results(testimony_html)
    print(f"Parsed {len(speech_items)} speech links from {speeches_html.name}")
    print(f"Parsed {len(testimony_items)} testimony links from {testimony_html.name}")

    all_items = speech_items + testimony_items

    last_date = get_last_date_in_csv(CSV_PATH)
    if last_date:
        print(f"Most recent date already in CSV: {last_date.isoformat()}")

    seen_links = load_seen_links(SEEN_LINKS_PATH)

    # Filter: after last_date already in CSV, and not already scraped.
    to_scrape = []
    for date_str, link in all_items:
        if link in seen_links:
            continue
        try:
            item_date = datetime.datetime.strptime(date_str, "%m/%d/%Y").date()
        except ValueError:
            print(f"  ! Unparseable date '{date_str}' for {link}, keeping just in case", file=sys.stderr)
            item_date = None
        if last_date and item_date and item_date <= last_date:
            continue
        to_scrape.append((item_date, link))

    # Dedupe (a link could theoretically appear on both pages) and sort
    # oldest-first so ids come out in chronological order.
    seen_in_batch = set()
    deduped = []
    for item_date, link in to_scrape:
        if link in seen_in_batch:
            continue
        seen_in_batch.add(link)
        deduped.append((item_date, link))
    deduped.sort(key=lambda x: (x[0] is None, x[0]))

    if not deduped:
        print("No backfill items found. Nothing to do.")
        return

    print(f"Backfilling {len(deduped)} item(s), oldest first:")
    for item_date, link in deduped:
        print(f"  - {item_date}: {link}")

    next_id = get_next_id(CSV_PATH)
    new_rows = []
    newly_seen_links = []
    failed = []  # (item_date, link) pairs that failed after retries

    for i, (item_date, link) in enumerate(deduped):
        print(f"[{i + 1}/{len(deduped)}] Scraping {link}")
        scraped = scrape_speech_page(link)

        if not scraped["ok"]:
            # Don't write a row, don't mark seen — leave it for a rerun
            # instead of burning an id on a blank row we can't recover.
            failed.append((item_date, link))
            if i < len(deduped) - 1:
                time.sleep(REQUEST_DELAY_SECONDS)
            continue

        date = normalize_date(scraped["date"]) if scraped["date"] else (
            item_date.strftime("%Y-%m-%d") if item_date else ""
        )

        if not scraped["content"]:
            print(f"  ! Warning: no content extracted for {link}", file=sys.stderr)

        new_rows.append({
            "id": next_id,
            "date": date,
            "title": scraped["title"],
            "speaker": scraped["speaker"],
            "content": scraped["content"],
        })
        newly_seen_links.append(link)
        next_id += 1

        if i < len(deduped) - 1:
            time.sleep(REQUEST_DELAY_SECONDS)

    if new_rows:
        append_rows_to_csv(CSV_PATH, new_rows)
        append_seen_links(SEEN_LINKS_PATH, newly_seen_links)
        print(f"Appended {len(new_rows)} new row(s) to {CSV_PATH}")

    if failed:
        print(f"\n{len(failed)} link(s) failed after retries and were skipped (not written, not marked seen):", file=sys.stderr)
        for item_date, link in failed:
            print(f"  - {item_date}: {link}", file=sys.stderr)
        print("Just rerun this script — they're still in the source HTML and not in .seen_links.txt, so they'll be picked up again.", file=sys.stderr)


if __name__ == "__main__":
    main()

Parsed 58 speech links from Federal Reserve Board - 2026 Speeches.html
Parsed 4 testimony links from Federal Reserve Board - Testimonies Only.html
Most recent date already in CSV: 2026-05-27
Backfilling 4 item(s), oldest first:
  - 2026-05-29: https://www.federalreserve.gov/newsevents/speech/bowman20260529a.htm
  - 2026-05-31: https://www.federalreserve.gov/newsevents/speech/powell20260531a.htm
  - 2026-06-04: https://www.federalreserve.gov/newsevents/testimony/bowman20260604a.htm
  - 2026-06-06: https://www.federalreserve.gov/newsevents/speech/barr20260606a.htm
[1/4] Scraping https://www.federalreserve.gov/newsevents/speech/bowman20260529a.htm
[2/4] Scraping https://www.federalreserve.gov/newsevents/speech/powell20260531a.htm
[3/4] Scraping https://www.federalreserve.gov/newsevents/testimony/bowman20260604a.htm
[4/4] Scraping https://www.federalreserve.gov/newsevents/speech/barr20260606a.htm
Appended 4 new row(s) to dataset\fed_speech.csv
